<!-- @format -->

# Vanilla Autoencoder on Fashion-MNIST

Trains a `784 → 256 → 64 → 256 → 784` autoencoder with a 64-dimensional
latent space. The best checkpoint (lowest validation MSE) is reused by
Experiment 3 in `08_classifier_analysis.ipynb`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

while not (
    (PROJECT_ROOT / "data_loader.py").is_file()
    and (PROJECT_ROOT / "models" / "mlp_classifier.py").is_file()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import json
import time

import matplotlib.pyplot as plt
import torch
from torch import nn, optim

from data_loader import get_dataloaders
from models.autoencoder import build_vanilla_autoencoder
from utils.metrics import psnr, ssim
from utils.paths import (
    CHECKPOINT_DIR,
    DATA_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    ensure_results_dirs,
    get_device,
)
from utils.reproducibility import set_seed


In [ ]:
SEED = 42
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.001
LATENT_DIM = 64

DEVICE = get_device()
ensure_results_dirs()

print(f"Device: {DEVICE}")


In [ ]:
set_seed(SEED)

train_loader, val_loader, test_loader = get_dataloaders(
    root=DATA_DIR, batch_size=BATCH_SIZE, seed=SEED
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Testing samples: {len(test_loader.dataset)}")


<!-- @format -->

## Model and optimizer


In [ ]:
model = build_vanilla_autoencoder(LATENT_DIM).to(DEVICE)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)


<!-- @format -->

## Training


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.set_grad_enabled(is_training):
        for images, _ in loader:
            images = images.to(DEVICE)
            images = images.flatten(start_dim=1)

            reconstruction = model(images)
            loss = criterion(reconstruction, images)

            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

    return total_loss / total_samples


In [ ]:
history = []

best_val_loss = float("inf")
checkpoint_path = CHECKPOINT_DIR / "vanilla_ae_best.pth"

for epoch in range(EPOCHS):
    start_time = time.perf_counter()

    train_loss = run_epoch(model, train_loader, criterion, optimizer)
    val_loss = run_epoch(model, val_loader, criterion)

    epoch_time = time.perf_counter() - start_time

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "epoch_time": epoch_time,
        }
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"| Train MSE: {train_loss:.6f} "
        f"| Val MSE: {val_loss:.6f} "
        f"| Time: {epoch_time:.2f}s"
    )


<!-- @format -->

## Test reconstruction


In [ ]:
model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
)
model.eval()

original_images = []
reconstructed_images = []

with torch.inference_mode():
    for images, _ in test_loader:
        images_device = images.to(DEVICE).flatten(start_dim=1)
        reconstruction = model(images_device)

        original_images.append(images_device.cpu())
        reconstructed_images.append(reconstruction.cpu())

original_images = torch.cat(original_images).to(DEVICE)
reconstructed_images = torch.cat(reconstructed_images).to(DEVICE)

test_loss = criterion(reconstructed_images, original_images).item()
test_mse = torch.mean((original_images - reconstructed_images) ** 2).item()
test_psnr = psnr(reconstructed_images, original_images).item()
test_ssim = ssim(reconstructed_images, original_images).item()

print("FINAL TEST RESULTS")
print(f"Best Validation MSE: {best_val_loss:.6f}")
print(f"Test MSE:            {test_mse:.6f}")
print(f"Test PSNR:           {test_psnr:.2f} dB")
print(f"Test SSIM:           {test_ssim:.4f}")


In [ ]:
num_samples = 8

fig, axes = plt.subplots(2, num_samples, figsize=(16, 5))

for i in range(num_samples):
    original = original_images[i].reshape(28, 28).cpu().numpy()
    reconstruction = reconstructed_images[i].reshape(28, 28).cpu().numpy()

    axes[0, i].imshow(original, cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(reconstruction, cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Reconstructed", fontsize=10)

fig.suptitle("Vanilla Autoencoder Reconstruction", fontsize=14)
plt.tight_layout()

figure_path = FIGURES_DIR / "vanilla_ae_reconstructions.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved:", figure_path)


<!-- @format -->

## Save metrics


In [ ]:
results = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "latent_dim": LATENT_DIM,
    "best_validation_mse": best_val_loss,
    "test_mse": test_mse,
    "test_psnr": test_psnr,
    "test_ssim": test_ssim,
    "history": history,
}

metrics_path = METRICS_DIR / "vanilla_ae.json"

with open(metrics_path, "w") as f:
    json.dump(results, f, indent=4)

print("Metrics saved:", metrics_path)
